In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ========================================
# Store Sales - Rolling 实验（基于 safe lag 主版本）
# 说明：
# 1. 本 notebook 在 safe lag 主版本基础上加入 safe rolling_mean_7
# 2. 当前实验特征：基础特征 + lag_16 + lag_21 + lag_28 + rolling_mean_7_from_shift16
# 3. 当前验证分数约为：0.450380
# 4. 当前结论：未优于 safe lag 主版本，暂不保留
# ========================================

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

# ========================================
# 0. 读取数据
# 这份 notebook 只保留 rolling 实验真正需要的数据
# ========================================
PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train = pd.read_csv(PATH + "train.csv", parse_dates=["date"])
test = pd.read_csv(PATH + "test.csv", parse_dates=["date"])
stores = pd.read_csv(PATH + "stores.csv")
oil = pd.read_csv(PATH + "oil.csv", parse_dates=["date"])

print("train shape:", train.shape)
print("test shape:", test.shape)

# ========================================
# 1. 搭数据底座：先合并 stores 和 oil
# ========================================
base_train = train.merge(stores, on="store_nbr", how="left")
base_test = test.merge(stores, on="store_nbr", how="left")

base_train = base_train.merge(oil, on="date", how="left")
base_test = base_test.merge(oil, on="date", how="left")

# 处理 oil 缺失值
base_train = base_train.sort_values("date").copy()
base_test = base_test.sort_values("date").copy()

base_train["dcoilwtico"] = base_train["dcoilwtico"].ffill().bfill()
base_test["dcoilwtico"] = base_test["dcoilwtico"].ffill().bfill()

print("base_train oil missing:", base_train["dcoilwtico"].isna().sum())
print("base_test oil missing:", base_test["dcoilwtico"].isna().sum())

# ========================================
# 2. 复制工作表并构造基础日期特征
# ========================================
train_df = base_train.copy()
test_df = base_test.copy()

for df in [train_df, test_df]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# ========================================
# 3. 定义基础特征列
# ========================================
cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]

num_cols = [
    "onpromotion",
    "dcoilwtico",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear",
    "is_weekend"
]

feature_cols = cat_cols + num_cols
target_col = "sales"

for col in cat_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

# ========================================
# 4. 构造 safe lag + safe rolling 特征
#
# lag:
# - lag_16
# - lag_21
# - lag_28
#
# rolling:
# - rolling_mean_7_from_shift16
#
# rolling 的构造方式：
# 先按 store_nbr + family 分组、按 date 排序，
# 对 sales 先 shift(16)，再 rolling(7).mean()
# ========================================
lag_df = pd.concat([
    train_df[["id", "date", "store_nbr", "family", "sales"]].copy(),
    test_df[["id", "date", "store_nbr", "family"]].assign(sales=np.nan).copy()
], ignore_index=True)

lag_df = lag_df.sort_values(["store_nbr", "family", "date"]).copy()

lag_df["lag_16"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(16)
lag_df["lag_21"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(21)
lag_df["lag_28"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(28)

lag_df["rolling_mean_7_from_shift16"] = (
    lag_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(16).rolling(window=7, min_periods=7).mean())
)

train_df_feat = train_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28", "rolling_mean_7_from_shift16"]],
    on="id",
    how="left"
)

test_df_feat = test_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28", "rolling_mean_7_from_shift16"]],
    on="id",
    how="left"
)

print("train feature missing:")
print(train_df_feat[["lag_16", "lag_21", "lag_28", "rolling_mean_7_from_shift16"]].isna().sum())

print("test feature missing:")
print(test_df_feat[["lag_16", "lag_21", "lag_28", "rolling_mean_7_from_shift16"]].isna().sum())

# 缺失值先统一填 0
fill_cols = ["lag_16", "lag_21", "lag_28", "rolling_mean_7_from_shift16"]

for col in fill_cols:
    train_df_feat[col] = train_df_feat[col].fillna(0)
    test_df_feat[col] = test_df_feat[col].fillna(0)

feature_cols_rolling = feature_cols + fill_cols

# ========================================
# 5. 只保留最近 365 天数据
# ========================================
cutoff_date = train_df_feat["date"].max() - pd.Timedelta(days=365)
train_df_small = train_df_feat[train_df_feat["date"] >= cutoff_date].copy()
train_df_small = train_df_small.sort_values(["store_nbr", "family", "date"]).copy()

# ========================================
# 6. 按时间切分验证集（最后 16 天）
# ========================================
last_date = train_df_small["date"].max()
val_start_date = last_date - pd.Timedelta(days=15)

train_part = train_df_small[train_df_small["date"] < val_start_date].copy()
valid_part = train_df_small[train_df_small["date"] >= val_start_date].copy()

print("train_part date range:", train_part["date"].min(), "to", train_part["date"].max())
print("valid_part date range:", valid_part["date"].min(), "to", valid_part["date"].max())

# ========================================
# 7. 构造训练 / 验证 / 测试输入
# ========================================
X_train = train_part[feature_cols_rolling].copy()
X_valid = valid_part[feature_cols_rolling].copy()
X_test = test_df_feat[feature_cols_rolling].copy()

y_train = np.log1p(train_part[target_col].values)
y_valid = np.log1p(valid_part[target_col].values)

print("当前训练特征列：")
print(X_train.columns.tolist())

# ========================================
# 8. 训练 CatBoost 模型
# ========================================
model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=50
)

# ========================================
# 9. 在验证集上评估
# ========================================
valid_pred_log = model.predict(X_valid)
valid_pred = np.expm1(valid_pred_log)
valid_pred = np.clip(valid_pred, 0, None)

valid_true = valid_part[target_col].values

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

score = rmsle(valid_true, valid_pred)
print("Validation RMSLE:", score)

# ========================================
# 10. 可选：导出 rolling 实验版本 submission
# 这版最终没有保留为主版本，但为了完整性仍可导出
# ========================================
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    "id": test_df["id"],
    "sales": test_pred
})

print(submission.head())

submission.to_csv("submission_safe_lag_plus_safe_rolling_mean_7.csv", index=False)
print("submission_safe_lag_plus_safe_rolling_mean_7.csv saved!")

# Store Sales - Rolling 实验（基于 safe lag 主版本）

## 实验目标
本实验在当前主版本 **safe lag（lag_16 / lag_21 / lag_28）** 的基础上，尝试加入一个新的时间序列特征：

- `rolling_mean_7_from_shift16`

目标是检查：  
在当前验证设定下，加入这个 **safe rolling** 特征后，模型是否能比当前主版本进一步提升。

---

## 当前实验版本
**实验名：** `safe_lag_plus_safe_rolling_mean_7_catboost_recent365`

### 使用特征
- stores
- oil
- 基础日期特征
- `lag_16`
- `lag_21`
- `lag_28`
- `rolling_mean_7_from_shift16`

### 验证方式
- 仅使用最近 365 天数据
- 最后 16 天作为 validation
- 不使用随机 KFold

### 模型
- `CatBoostRegressor`

### 参数
- `iterations=300`
- `depth=6`
- `learning_rate=0.05`

---

## rolling 特征构造说明
本次使用的 rolling 特征不是普通 `rolling_mean_7`，而是 **safe rolling_mean_7**。

构造方式如下：

1. 按 `store_nbr + family` 分组  
2. 按 `date` 排序  
3. 对 `sales` 先做 `shift(16)`  
4. 再做 `rolling(7).mean()`  

得到：

- `rolling_mean_7_from_shift16`

### 为什么这样做？
因为当前验证集是最后 16 天。  
如果直接做普通 rolling，很可能会读到验证窗口内部的真实销量，导致结果偏乐观。

而本次做法先 `shift(16)`，再做 7 天 rolling 均值，表示当前样本只使用 **至少 16 天以前** 的历史销量信息，因此在当前验证设定下相对更安全、更可信。

---

## 实验结果
**Validation RMSLE = 0.450380**

---

## 与当前主版本对比

### 当前主版本
`safe_lag_16_21_28_catboost_recent365`

- Validation RMSLE = `0.447398`

### 本次 rolling 版本
`safe_lag_plus_safe_rolling_mean_7_catboost_recent365`

- Validation RMSLE = `0.450380`

### 分数变化
加入 `rolling_mean_7_from_shift16` 后，分数变差了约：

- `0.450380 - 0.447398 = 0.002982`

这说明本次 rolling 特征在当前设定下 **没有带来提升**。

---

## 结果解释
本次 rolling 实验有两个重要结论：

1. 这次 rolling 特征的构造方式相对安全，不属于明显的 leakage 型实验结果。  
2. 即使在相对可信的验证设定下，`rolling_mean_7_from_shift16` 仍然没有优于当前 safe lag 主版本。

一个可能原因是：

- 当前 `lag_16`、`lag_21`、`lag_28` 已经捕捉了较多历史信号；
- 额外加入一个经过 `shift(16)` 的 7 天 rolling 均值后，并没有提供足够新的有效信息；
- 也可能是这个 rolling 均值距离目标日期较远，信息过于粗糙，未能帮助当前模型。

---

## 最终结论
**决策：不保留 `safe rolling_mean_7_from_shift16` 为当前主版本。**

当前最佳版本仍然是：

- `safe_lag_16_21_28_catboost_recent365`
- Validation RMSLE = `0.447398`

---

## 当前项目状态
到本周为止，项目已经完成：

- 数据理解与初始清洗
- baseline 建模
- short lag 实验
- safe lag 实验
- holiday 实验
- safe rolling_mean_7 实验
- rolling 特征是否保留的决策

---

## 本实验的价值
虽然本次 rolling 特征没有提升分数，但这次实验仍然有价值：

- 进一步强化了时间序列实验中对 **验证可信度** 的重视
- 说明“加新特征”不等于一定有效
- 说明当前阶段应该优先保持验证方式稳定，再继续推进更有针对性的特征工程